In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window


In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','gross_price','Data source')
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(catalog,data_source)

In [0]:
df = (
    spark.read.format('csv')
    .option('inferSchmea','true')
    .option('header','true')
    .load('/Volumes/fmcg/bronze/source_fmcg/gross_price/')
    .withColumn('read_timestamp',F.current_timestamp())
    .select("*","_metadata.file_name",'_metadata.file_size')
)

In [0]:
df.display()

In [0]:
df.write.format('delta')\
    .option('delta.enableChangeDataFeed','true')\
    .mode('overwrite')\
        .saveAsTable('fmcg.bronze.gross_price')

# **Silver Tranformation**

In [0]:
df_bronze = spark.sql('select * from fmcg.bronze.gross_price')
df_bronze.display()

Normalization "Month" field

In [0]:
df_bronze.select('month').distinct().display()

In [0]:
# 1️. Parse `month` from multiple possible formats

date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]
df_silver = df_bronze.withColumn( "month", F.coalesce( F.try_to_date(F.col("month"), "yyyy/MM/dd"), F.try_to_date(F.col("month"), "dd/MM/yyyy"), F.try_to_date(F.col("month"), "yyyy-MM-dd"), F.try_to_date(F.col("month"), "dd-MM-yyyy") ) )

In [0]:
df_silver.select('month').distinct().display()

unified and handle the gross price

In [0]:
                                      
df_silver = df_silver.withColumn( "gross_price", F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double")) .otherwise(F.col("gross_price").cast("double"))) .otherwise(0))

df_silver.display(10)

In [0]:
df_products = spark.table('fmcg.silver.products')
df_joined = df_silver.join(df_products.select('product_id','product_code'), on = 'product_id',how = 'inner')
df_joined = df_joined.select('product_id','product_code','month','gross_price','read_timestamp','file_name','file_size')
df_joined.show(5)

In [0]:
df_joined.write\
    .format('delta')\
        .option('delta.enableChangeDataFeed','true')\
            .option('mergeSchema','true')\
                .mode('overwrite')\
                    .saveAsTable('fmcg.silver.gross_price')

In [0]:
df_silver = spark.sql('select * from fmcg.silver.gross_price')

In [0]:
df_gold_price = df_silver.select('product_code','month','gross_price')
df_gold_price.display(5)

In [0]:
df_gold_price = (
    df_gold_price
    .withColumn('year',F.year('month'))
    .withColumn('is_zero',F.when(F.col('gross_price') == 0,1).otherwise(0))
)

w = (
    Window
    .partitionBy('product_code','year')
    .orderBy(F.col('is_zero'),F.col('month').desc())
)

df_gold_latest_price = (
    df_gold_price
    .withColumn('rnk',F.row_number().over(w))
    .filter(F.col('rnk') == 1)
)
df_gold_latest_price.display()

In [0]:
df_gold_latest_price = df_gold_latest_price.select('product_code','year','gross_price').withColumnRenamed('gross_price','price_inr').select('product_code','price_inr','year')

#change year to string

df_gold_latest_price = df_gold_latest_price.withColumn('year',F.col('year').cast('string'))

df_gold_latest_price.show(5)

In [0]:
df_gold_latest_price.printSchema()

In [0]:
delta_table = DeltaTable.forName(spark,'fmcg.gold.dim_gross_price')
delta_table.alias('target').merge(
    source = df_gold_latest_price.alias('source'),
    condition = 'target.product_code =  source.product_code'
).whenMatchedUpdate(
    set = {
        'price_inr' : 'source.price_inr',
        'year':'source.year'

    }
).whenNotMatchedInsert(
    values = {
        'product_code':'source.product_code',
        'price_inr':'source.price_inr'
    }
).execute()